# Lakelink Fiber Provable Resilience: v4 KPI delta attribution writer

**Owner:** @data-analyst. **Tasks:** P38-DA-V4-KPI-DELTA-ATTRIBUTION (INTERIM build), P40-DA-V4-WRITERS-CHECKIN (this check-in), P39-DA-V4-KPI-DELTA-FINALIZE (FINAL build).

**Write target (only):** `cdm_tmforum.provable_resilience_intelligence.kpi_delta_attribution_v4_candidate`

This notebook compares the frozen v3 objects with the v4-candidate shadow objects and attributes each KPI change to a driver:

| driver_code | Change | Source objects |
| --- | --- | --- |
| HSI_FIX | Hazard-span intersection rebuild (line distance at 4 dp, future-hazard exclusion) | resilience_score_snapshot_v4_candidate |
| BB_SEGMENTATION | Backbone segmentation at parent-route grain (derived_shared_route SRLGs) | unified_shared_risk_group_v4_candidate |
| SRLG_REDERIVE | SRLG re-derive (conduit/site/geo) and path_diversity rescore | unified_shared_risk_group_v4_candidate |
| DEDUP | Revenue Protected dedup to one row per service | mv_mrc_protected_v4_candidate |
| TOTAL | Net v3 to v4-candidate delta | all |

**Freeze rules (DEC-PRI-V4-SHADOW-BUILD):**
* Every v3 object is READ only (the snapshot is pinned with `VERSION AS OF 22`).
* The only write is to `kpi_delta_attribution_v4_candidate`; cell 2 asserts this.
* `dry_run` defaults to `true`, so running the notebook computes and checks the KPIs without writing. Set `dry_run=false` only in a @pm-approved write window.
* If the candidate snapshot shows rescored path diversity (BUG-DP-070 and ML P36 have landed), the notebook stops. The sequential-substitution split for BB_SEGMENTATION and SRLG_REDERIVE is added under P39.

In [0]:
from datetime import datetime, timezone

CATALOG_SCHEMA = "cdm_tmforum.provable_resilience_intelligence"
OUTPUT_TABLE = f"{CATALOG_SCHEMA}.kpi_delta_attribution_v4_candidate"
V3_SNAPSHOT_VERSION = 22

dbutils.widgets.dropdown("dry_run", "true", ["true", "false"], "Dry run (no write)")
dbutils.widgets.dropdown("report_status", "INTERIM", ["INTERIM", "FINAL"], "Report status")
dbutils.widgets.text("report_version", "interim-1 (P38 DA 2026-10-26)", "Report version")

DRY_RUN = dbutils.widgets.get("dry_run") == "true"
REPORT_STATUS = dbutils.widgets.get("report_status")
REPORT_VERSION = dbutils.widgets.get("report_version")

# Freeze guard: the only object this notebook may write is a *_v4_candidate table.
assert OUTPUT_TABLE.endswith("_v4_candidate"), f"Refusing to write a non-candidate object: {OUTPUT_TABLE}"

# v3 baselines (read only)
V3_SNAPSHOT = f"{CATALOG_SCHEMA}.resilience_score_snapshot VERSION AS OF {V3_SNAPSHOT_VERSION}"
V3_SRLG = f"{CATALOG_SCHEMA}.unified_shared_risk_group"
V3_ACTION_QUEUE_MV = f"{CATALOG_SCHEMA}.action_queue_status"
V3_PREVENTED_MV = f"{CATALOG_SCHEMA}.prevented_outages"
# v4 candidates (read only here)
C_SNAPSHOT = f"{CATALOG_SCHEMA}.resilience_score_snapshot_v4_candidate"
C_SRLG = f"{CATALOG_SCHEMA}.unified_shared_risk_group_v4_candidate"
C_MRC_MV = f"{CATALOG_SCHEMA}.mv_mrc_protected_v4_candidate"

CRITICAL_THRESHOLD = 40  # overall_score < 40 = critical
print(f"target={OUTPUT_TABLE} dry_run={DRY_RUN} status={REPORT_STATUS} version={REPORT_VERSION}")

In [0]:
def scalar(sql):
    """Run a read-only query that returns one value."""
    return spark.sql(sql).first()[0]

# Latest v3 row per service at the pinned version, and the candidate (already one row per service).
V3_LATEST = f"""(SELECT * FROM {V3_SNAPSHOT}
  QUALIFY ROW_NUMBER() OVER (PARTITION BY service_id ORDER BY score_time DESC, created_at DESC) = 1)"""

def snapshot_kpis(src):
    r = spark.sql(f"""
      SELECT count(*) AS services,
             round(avg(overall_score), 2) AS avg_score,
             count_if(overall_score < {CRITICAL_THRESHOLD}) AS critical_count,
             round(sum(revenue_at_risk_usd)) AS revenue_at_risk,
             count_if(revenue_at_risk_usd > 0) AS services_with_rar,
             round(100 * avg(CASE WHEN srlg_count = 0 THEN 1 ELSE 0 END), 2) AS diversity_coverage_pct,
             count_if(srlg_count > 0) AS shared_risk_services,
             round(sum(CASE WHEN srlg_count > 0 THEN revenue_at_risk_usd ELSE 0 END)) AS false_diversity_revenue,
             round(avg(hazard_exposure_score), 2) AS avg_hazard_component,
             sum(active_hazards) AS active_hazard_links
      FROM {src}""").first().asDict()
    return r

def srlg_kpis(src):
    return spark.sql(f"""
      SELECT count(DISTINCT srlg_id) AS srlg_count,
             count(DISTINCT CASE WHEN severity = 'critical' THEN srlg_id END) AS critical_srlg_count,
             count(DISTINCT span_id) AS spans_exposed,
             count(DISTINCT conduit_id) AS conduits_exposed
      FROM {src}""").first().asDict()

def srlg_by_source(src):
    rows = spark.sql(f"""
      SELECT source, count(DISTINCT srlg_id) AS n,
             count(DISTINCT CASE WHEN severity = 'critical' THEN srlg_id END) AS crit,
             count(DISTINCT span_id) AS spans
      FROM {src} GROUP BY source""").collect()
    return {r.source: r.asDict() for r in rows}

In [0]:
v3 = snapshot_kpis(V3_LATEST)
v4 = snapshot_kpis(C_SNAPSHOT)

# Critical-service MRC (distinct service). mrc_usd lives on the candidate; v3_overall_score is the pinned v3 score.
v3["critical_mrc"] = scalar(f"SELECT round(sum(CASE WHEN v3_overall_score < {CRITICAL_THRESHOLD} THEN mrc_usd ELSE 0 END)) FROM {C_SNAPSHOT}")
v4["critical_mrc"] = scalar(f"SELECT round(sum(CASE WHEN overall_score < {CRITICAL_THRESHOLD} THEN mrc_usd ELSE 0 END)) FROM {C_SNAPSHOT}")

# Critical transitions (newly critical by v3 band) and their revenue at risk
trans = spark.sql(f"""
  SELECT v3_risk_band, count(*) AS n, round(sum(revenue_at_risk_usd)) AS rar
  FROM {C_SNAPSHOT}
  WHERE overall_score < {CRITICAL_THRESHOLD} AND v3_overall_score >= {CRITICAL_THRESHOLD}
  GROUP BY v3_risk_band""").collect()
newly_critical = {r.v3_risk_band: r.n for r in trans}
newly_critical_rar = sum(r.rar or 0 for r in trans)
left_critical = scalar(f"SELECT count_if(v3_overall_score < {CRITICAL_THRESHOLD} AND overall_score >= {CRITICAL_THRESHOLD}) FROM {C_SNAPSHOT}")
hazard_changed = scalar(f"SELECT count_if(active_hazards <> v3_active_hazards) FROM {C_SNAPSHOT}")

# SRLG layer
v3_srlg, v4_srlg = srlg_kpis(V3_SRLG), srlg_kpis(C_SRLG)
v3_src, v4_src = srlg_by_source(V3_SRLG), srlg_by_source(C_SRLG)
BB_SOURCES = {"derived_shared_route"}
def src_delta(key, sources):
    return sum(v4_src.get(s, {}).get(key, 0) - v3_src.get(s, {}).get(key, 0) for s in sources)
rederive_sources = (set(v3_src) | set(v4_src)) - BB_SOURCES - {"seed"}

# Revenue Protected (metric views need MEASURE())
rp_legacy_aq = scalar(f"SELECT round(MEASURE(`Revenue Protected USD`)) FROM {V3_ACTION_QUEUE_MV}")
rp_legacy_po = scalar(f"SELECT round(MEASURE(revenue_protected)) FROM {V3_PREVENTED_MV}")
sp_legacy_po = scalar(f"SELECT MEASURE(services_protected) FROM {V3_PREVENTED_MV}")
mrc = spark.sql(f"""SELECT round(MEASURE(`MRC Protected USD`)) AS rp, MEASURE(`Services Protected`) AS sp,
  round(MEASURE(`MRC Protected Proactive Only USD`)) AS rp_pro, MEASURE(`Services Protected Proactive Only`) AS sp_pro
  FROM {C_MRC_MV}""").first()

print("v3:", v3); print("v4:", v4)
print("SRLG v3:", v3_srlg, "v4:", v4_srlg)
print("newly critical:", newly_critical, newly_critical_rar, "left critical:", left_critical)
print("revenue protected:", rp_legacy_aq, rp_legacy_po, sp_legacy_po, mrc)

In [0]:
# BB_SEGMENTATION and SRLG_REDERIVE only change service scores after BUG-DP-070 and the ML P36 rescore.
# Until then the candidate snapshot carries the v3 path_diversity_score and srlg_count.
path_rescored = scalar(f"""
  SELECT count(*) FROM {C_SNAPSHOT} c JOIN {V3_LATEST} v USING (service_id)
  WHERE c.path_diversity_score <> v.path_diversity_score OR c.srlg_count <> v.srlg_count""")
PATH_DRIVERS_LANDED = path_rescored > 0
print(f"services with rescored path diversity / srlg_count: {path_rescored}")

if PATH_DRIVERS_LANDED:
    raise NotImplementedError(
        f"{path_rescored} services have rescored path diversity. Do not attribute the whole score delta to "
        "HSI_FIX. Add the sequential-substitution split for BB_SEGMENTATION and SRLG_REDERIVE "
        "(P39-DA-V4-KPI-DELTA-FINALIZE) before writing.")
assert REPORT_STATUS == "INTERIM", "FINAL needs the P39 path-driver split"

In [0]:
DRIVERS = {
    "HSI_FIX": ("Hazard-span intersection rebuild (line distance at 4 dp, future-hazard exclusion)",
                "BUG-QA-064,BUG-QA-065,BUG-QA-066,P36-DA-FUTURE-HAZARD-KPI-AUDIT"),
    "BB_SEGMENTATION": ("Backbone segmentation at parent-route grain (derived_shared_route SRLGs)",
                        "BUG-QA-067a,BUG-DP-068,BUG-DP-069,BUG-QA-073"),
    "SRLG_REDERIVE": ("SRLG re-derive (conduit/site/geo grouping) and path_diversity rescore",
                      "BUG-QA-074,BUG-DP-070,P36-ML-V4-RESCORE-CONDUIT-DIVERSITY"),
    "DEDUP": ("Revenue Protected dedup to one row per service", "P29-DA-REV-PROTECTED-DEDUP"),
    "TOTAL": ("Net v3 to v4-candidate delta", ""),
}
SEQ = "sequential substitution (v3 then swap driver input)"
SNAP_V3 = f"resilience_score_snapshot (v{V3_SNAPSHOT_VERSION})"
SNAP_V4 = "resilience_score_snapshot_v4_candidate"
PENDING = ("Candidate snapshot still carries v3 path_diversity_score and srlg_count, so the delta is 0 "
           "until ML P36 rescores the affected services (blocked on BUG-DP-070).")
rows = []

def add(kpi, name, unit, driver, state, v3v, delta, v4v, method, v3s, v4s, notes):
    label, codes = DRIVERS[driver]
    rows.append(dict(kpi_code=kpi, kpi_name=name, kpi_unit=unit, driver_code=driver, driver_label=label,
                     driver_task_codes=codes, driver_state=state, v3_value=float(v3v),
                     delta_value=float(delta), v4_candidate_value=None if v4v is None else float(v4v),
                     attribution_method=method, v3_source=v3s, v4_source=v4s, notes=notes))

def snapshot_kpi(kpi, name, unit, key, hsi_note, dedup=False, hsi_state="landed_in_candidate", v3s=SNAP_V3, path_rows=True):
    a, b = v3[key], v4[key]
    d = round(b - a, 2)
    # While path drivers are pending, the full candidate delta is the HSI delta (asserted in the cell above).
    add(kpi, name, unit, "HSI_FIX", hsi_state, a, d, b, SEQ, v3s, SNAP_V4, hsi_note)
    if path_rows:
        add(kpi, name, unit, "BB_SEGMENTATION", "pending_upstream", a, 0, None, SEQ, v3s, SNAP_V4, PENDING)
        add(kpi, name, unit, "SRLG_REDERIVE", "pending_upstream", a, 0, None, SEQ, v3s, SNAP_V4, PENDING)
    if dedup:
        add(kpi, name, unit, "DEDUP", "not_applicable", a, 0, None, SEQ, v3s, SNAP_V4, "Dedup affects Revenue Protected only.")
    add(kpi, name, unit, "TOTAL", "total", a, d, b, "sum of landed drivers", v3s, SNAP_V4, f"{REPORT_STATUS}: HSI is the only landed driver.")

snapshot_kpi("AVG_SCORE", "Average resilience score", "score 0-100", "avg_score",
    f"Hazard exposure component average {v3['avg_hazard_component']} -> {v4['avg_hazard_component']}. "
    f"Active hazard-service links {v3['active_hazard_links']:,} -> {v4['active_hazard_links']:,}; "
    f"{hazard_changed:,} of {v4['services']:,} services changed hazard count.", dedup=True)
snapshot_kpi("CRITICAL_COUNT", f"Critical services (score < {CRITICAL_THRESHOLD})", "distinct services", "critical_count",
    f"Newly critical by v3 band: {newly_critical}; {left_critical} services leave critical. "
    f"Their revenue at risk is ${newly_critical_rar:,.0f}.")
snapshot_kpi("REVENUE_AT_RISK", "Revenue at risk (sum of revenue_at_risk_usd, one row per service)", "USD per month",
    "revenue_at_risk", f"Services with revenue at risk above 0: {v3['services_with_rar']:,} -> {v4['services_with_rar']:,}.")
snapshot_kpi("CRITICAL_MRC", "MRC of critical services (distinct service)", "USD per month", "critical_mrc",
    f"Sum of mrc_usd over distinct services with score < {CRITICAL_THRESHOLD}.", path_rows=False)
snapshot_kpi("DIVERSITY_COVERAGE_PCT", "Diversity coverage (active services with 0 shared SRLGs)", "percent",
    "diversity_coverage_pct", "Hazards do not change srlg_count.", hsi_state="not_applicable",
    v3s="diversity_coverage MV logic on " + SNAP_V3)
snapshot_kpi("FALSE_DIVERSITY_REVENUE", "Revenue at risk on false-diversity services", "USD per month",
    "false_diversity_revenue", f"Same {v4['shared_risk_services']:,} services; the change comes only from re-scored revenue_at_risk_usd.",
    v3s="diversity_coverage MV logic", path_rows=False)

# SRLG layer: per-source deltas (landed in candidate objects, not yet in service scores)
for kpi, name, key, crit_note in [
    ("SRLG_COUNT", "Distinct SRLGs (unified)", "n", None),
    ("CRITICAL_SRLG_COUNT", "Critical-severity SRLGs (unified)", "crit", True)]:
    total_key = "srlg_count" if key == "n" else "critical_srlg_count"
    a, b = v3_srlg[total_key], v4_srlg[total_key]
    bb, rd = src_delta(key, BB_SOURCES), src_delta(key, rederive_sources)
    rd_detail = ", ".join(f"{s} {v3_src.get(s, {}).get(key, 0)} -> {v4_src.get(s, {}).get(key, 0)}" for s in sorted(rederive_sources))
    add(kpi, name, "SRLGs", "BB_SEGMENTATION", "landed_in_candidate", a, bb, None, "per-source delta", "unified_shared_risk_group",
        "unified_shared_risk_group_v4_candidate", f"derived_shared_route: {bb} SRLGs covering {v4_src.get('derived_shared_route', {}).get('spans', 0):,} spans.")
    add(kpi, name, "SRLGs", "SRLG_REDERIVE", "landed_in_candidate", a, rd, None, "per-source delta", "unified_shared_risk_group",
        "unified_shared_risk_group_v4_candidate", rd_detail + ". Seed unchanged.")
    add(kpi, name, "SRLGs", "TOTAL", "total", a, b - a, b, "sum of drivers", "unified_shared_risk_group",
        "unified_shared_risk_group_v4_candidate", f"{REPORT_STATUS}; the candidate SRLG layer has landed but is not yet applied to service scores.")
add("SRLG_SPANS_EXPOSED", "Distinct spans in any SRLG (unified)", "spans", "TOTAL", "total", v3_srlg["spans_exposed"],
    v4_srlg["spans_exposed"] - v3_srlg["spans_exposed"], v4_srlg["spans_exposed"], "non-additive union", "unified_shared_risk_group",
    "unified_shared_risk_group_v4_candidate", f"Span coverage overlaps across sources, so it is not split by driver. Conduits {v3_srlg['conduits_exposed']} -> {v4_srlg['conduits_exposed']}.")
add("SHARED_RISK_SERVICES", "Services with srlg_count > 0", "distinct services", "TOTAL", "total", v3["shared_risk_services"],
    v4["shared_risk_services"] - v3["shared_risk_services"], v4["shared_risk_services"], "sum of landed drivers", SNAP_V3, SNAP_V4, f"{REPORT_STATUS}. {PENDING}")

# Revenue Protected dedup (replace measure)
add("REVENUE_PROTECTED", "Revenue Protected (action_queue_status legacy measure)", "USD per month", "DEDUP", "landed_in_candidate",
    rp_legacy_aq, mrc.rp - rp_legacy_aq, mrc.rp, "replace measure", "action_queue_status.Revenue Protected USD", "mv_mrc_protected_v4_candidate",
    f"Legacy sums revenue_at_risk per action row across all statuses (fan-out). v4 is the MRC of {mrc.sp:,} distinct services with at least one completed action.")
add("REVENUE_PROTECTED", "Revenue Protected (prevented_outages legacy measure)", "USD per month", "DEDUP", "landed_in_candidate",
    rp_legacy_po, mrc.rp - rp_legacy_po, mrc.rp, "replace measure", "prevented_outages.revenue_protected", "mv_mrc_protected_v4_candidate",
    f"Legacy sums MRC per proactive action row. Proactive-only v4 equivalent: ${mrc.rp_pro:,.0f} over {mrc.sp_pro:,} services.")
add("SERVICES_PROTECTED", "Services Protected", "distinct services", "DEDUP", "landed_in_candidate", sp_legacy_po, mrc.sp - sp_legacy_po,
    mrc.sp, "replace measure", "prevented_outages.services_protected", "mv_mrc_protected_v4_candidate",
    f"The v4 scope is any completed action, not only proactive (proactive-only is {mrc.sp_pro:,}).")

computed_at = datetime.now(timezone.utc).replace(tzinfo=None)
for r in rows:
    r.update(report_status=REPORT_STATUS, report_version=REPORT_VERSION, computed_at=computed_at, origin="MIXED")
print(f"{len(rows)} rows")

In [0]:
# Values published by P38 (interim-1, 2026-10-26). Drift is reported, not fatal: the SRLG candidates may be rebuilt
# by upstream tasks during the shadow window.
INTERIM_BASELINE = {
    ("AVG_SCORE", "TOTAL"): (61.55, 56.77), ("CRITICAL_COUNT", "TOTAL"): (848, 1087),
    ("REVENUE_AT_RISK", "TOTAL"): (48445093, 54440037), ("CRITICAL_MRC", "TOTAL"): (12791174, 16620967),
    ("DIVERSITY_COVERAGE_PCT", "TOTAL"): (74.19, 74.19), ("FALSE_DIVERSITY_REVENUE", "TOTAL"): (18041565, 19084500),
    ("SRLG_COUNT", "TOTAL"): (931, 1060), ("CRITICAL_SRLG_COUNT", "TOTAL"): (404, 525),
    ("SRLG_SPANS_EXPOSED", "TOTAL"): (9999, 15295), ("SHARED_RISK_SERVICES", "TOTAL"): (2065, 2065),
    ("SERVICES_PROTECTED", "DEDUP"): (2065, 4625),
}
import pandas as pd
pdf = pd.DataFrame(rows)
checks = []
for (kpi, drv), (e3, e4) in INTERIM_BASELINE.items():
    r = pdf[(pdf.kpi_code == kpi) & (pdf.driver_code == drv)].iloc[0]
    checks.append(dict(kpi=kpi, driver=drv, v3=r.v3_value, exp_v3=e3, v4=r.v4_candidate_value, exp_v4=e4,
                       match=abs(r.v3_value - e3) < 0.01 and abs(r.v4_candidate_value - e4) < 0.01))
checks = pd.DataFrame(checks)
display(checks)
print("drift:", checks[~checks.match][["kpi", "v4", "exp_v4"]].to_dict("records"))

In [0]:
from pyspark.sql import types as T

SCHEMA = T.StructType([
    T.StructField("kpi_code", T.StringType()), T.StructField("kpi_name", T.StringType()),
    T.StructField("kpi_unit", T.StringType()), T.StructField("driver_code", T.StringType()),
    T.StructField("driver_label", T.StringType()), T.StructField("driver_task_codes", T.StringType()),
    T.StructField("driver_state", T.StringType()), T.StructField("v3_value", T.DoubleType()),
    T.StructField("delta_value", T.DoubleType()), T.StructField("v4_candidate_value", T.DoubleType()),
    T.StructField("attribution_method", T.StringType()), T.StructField("v3_source", T.StringType()),
    T.StructField("v4_source", T.StringType()), T.StructField("notes", T.StringType()),
    T.StructField("report_status", T.StringType()), T.StructField("report_version", T.StringType()),
    T.StructField("computed_at", T.TimestampType()), T.StructField("origin", T.StringType()),
])
out_df = spark.createDataFrame([tuple(r[f.name] for f in SCHEMA.fields) for r in rows], SCHEMA)
display(out_df)

if DRY_RUN:
    print(f"DRY RUN: {out_df.count()} rows computed, nothing written to {OUTPUT_TABLE}.")
else:
    assert OUTPUT_TABLE.endswith("_v4_candidate")
    out_df.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(OUTPUT_TABLE)
    print(f"Wrote {out_df.count()} rows to {OUTPUT_TABLE}")

In [0]:
TABLE_COMMENT = (
    f"{REPORT_STATUS} (P38-DA-V4-KPI-DELTA-ATTRIBUTION, cutover gate 0). Lakelink Fiber Provable Resilience KPI delta report "
    f"comparing frozen v3 (resilience_score_snapshot v{V3_SNAPSHOT_VERSION} and legacy metric views) with the v4-candidate shadow "
    "objects. Grain: one row per KPI x driver; driver TOTAL holds the net delta. Drivers: HSI_FIX, BB_SEGMENTATION, SRLG_REDERIVE, "
    "DEDUP. Read-only on v3. Writer: notebooks/da_v4_kpi_delta_attribution. Finalised under P39-DA-V4-KPI-DELTA-FINALIZE.")
COLUMN_COMMENTS = {
    "kpi_code": "Stable KPI identifier (AVG_SCORE, CRITICAL_COUNT, REVENUE_AT_RISK, CRITICAL_MRC, DIVERSITY_COVERAGE_PCT, FALSE_DIVERSITY_REVENUE, SRLG_COUNT, CRITICAL_SRLG_COUNT, SRLG_SPANS_EXPOSED, SHARED_RISK_SERVICES, REVENUE_PROTECTED, SERVICES_PROTECTED)",
    "kpi_name": "Business-readable KPI name and definition",
    "kpi_unit": "Unit of v3_value, delta_value and v4_candidate_value",
    "driver_code": "Attribution driver: HSI_FIX, BB_SEGMENTATION, SRLG_REDERIVE, DEDUP, or TOTAL (net)",
    "driver_label": "Description of the driver change",
    "driver_task_codes": "Comma-separated Lakebase task codes that implement the driver",
    "driver_state": "landed_in_candidate = reflected in candidate objects; pending_upstream = not yet applied (delta reported as 0); not_applicable = driver does not affect KPI; total = net row",
    "v3_value": "KPI value on frozen v3 objects",
    "delta_value": "Change attributed to this driver (v4 minus v3 for TOTAL)",
    "v4_candidate_value": "KPI value after the driver (TOTAL row = full v4-candidate value); null when not separable",
    "attribution_method": "How the delta was isolated (sequential substitution, per-source delta, replace measure, non-additive union)",
    "v3_source": "v3 object(s) used for the baseline",
    "v4_source": "v4-candidate object(s) used",
    "notes": "Supporting evidence, sub-counts and caveats",
    "report_status": "INTERIM until all drivers land, then FINAL",
    "report_version": "Report build identifier",
    "computed_at": "UTC timestamp when the row was computed",
    "origin": "Data origin label (REAL/SYNTHETIC/MIXED); MIXED because it derives from synthetic-augmented inputs",
}

def q(s):
    return s.replace("'", "\\'")

if DRY_RUN:
    print(f"DRY RUN: {len(COLUMN_COMMENTS)} column comments and the table comment not applied.")
else:
    spark.sql(f"COMMENT ON TABLE {OUTPUT_TABLE} IS '{q(TABLE_COMMENT)}'")
    for col, txt in COLUMN_COMMENTS.items():
        spark.sql(f"ALTER TABLE {OUTPUT_TABLE} ALTER COLUMN {col} COMMENT '{q(txt)}'")
    # Tag mandate: telco_project = provable_resilience_intelligence. The UC tag policy currently rejects this value
    # (escalated); the failure is reported, not fatal.
    try:
        spark.sql(f"ALTER TABLE {OUTPUT_TABLE} SET TAGS ('telco_project' = 'provable_resilience_intelligence')")
    except Exception as e:
        print(f"Tag not applied: {str(e)[:200]}")
    print(f"Comments applied to {OUTPUT_TABLE}")